# 06 -- Reaction Reversibility Heuristics

This notebook ports
`ModelSEEDDatabase/Scripts/Thermodynamics/Estimate_Reaction_Reversibility.py`
into a parameterizable library and exercises every suggestion in
`/scratch/ctaylor/Reaction_Reversibility_Heuristics_Review.md`
against the 100-model panel from
`results/selected_ids.txt`.

Each variant cell reports two things:

1. **How many reactions changed direction** vs the baseline
   cascade, broken down by transition (`>` -> `=`, `=` -> `<`, ...).
2. **How that change propagates to FBA growth** on the 100 panel
   models -- how many models flip between grower and non-grower,
   and how many have a meaningful flux delta.

Caching: every heavy output lives in `notebooks/.kbcache/` via
`KBUtils_local`'s `NotebookSession`.  The 56K-reaction MSDB load,
the per-variant cascade, and the per-variant panel FBA are all
cached -- re-running the notebook should be sub-second after the
first execution.

**What does not change.**  Nothing under `ModelSEEDDatabase/` or
`core_models_kegg2/` is mutated.  All variant bounds live in
in-memory cobra models.

In [1]:
from pathlib import Path
import sys, copy, time, json, csv, hashlib

PROJECT_ROOT = Path('/scratch/ctaylor/core_models_analysis')
MSDB_ROOT    = Path('/scratch/ctaylor/ModelSEEDDatabase')
REPORTS      = PROJECT_ROOT / 'reports'
RESULTS      = PROJECT_ROOT / 'results'
SCRIPTS      = PROJECT_ROOT / 'scripts'

# Make the project scripts and MSDB BiochemPy importable.
sys.path.insert(0, str(SCRIPTS))
sys.path.insert(0, str(MSDB_ROOT / 'Libs' / 'Python'))

# KBUtils NotebookSession holds the .kbcache/ alongside this notebook.  The
# 56K-row MSDB reactions dict + the panel growth tables are cached here so
# re-runs skip the heavy load + FBA solves.
from kbutillib.notebook import NotebookSession
session = NotebookSession.for_notebook(
    notebook_file=__file__ if '__file__' in dir() else None,
    project_name='core_models_analysis',
)
print('Cache directory:', session.kbcache_dir)
print('Notebook name :', session.notebook_name)

import reversibility_lib as lib
import growth_heuristics as gh

Cache directory: /scratch/ctaylor/core_models_analysis/notebooks/.kbcache
Notebook name : None


[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


## 1. Load MSDB reactions + panel + on-disk FBA results

In [2]:
# Load the MSDB reaction database via BiochemPy.  Cached in the
# NotebookSession blob store the first time and reloaded as JSON on
# subsequent runs.
CACHE_KEY_MSDB_RXNS = 'msdb_reactions_v1'

def _load_msdb_reactions():
    from BiochemPy import Reactions
    return Reactions().loadReactions()

try:
    msdb_rxns = session.cache.load(CACHE_KEY_MSDB_RXNS)
    print(f'Loaded {len(msdb_rxns)} MSDB reactions from cache')
except KeyError:
    t = time.time()
    msdb_rxns = _load_msdb_reactions()
    session.cache.save(CACHE_KEY_MSDB_RXNS, msdb_rxns, type_hint='dict',
                       metadata={'source': 'BiochemPy.Reactions().loadReactions()'})
    print(f'Loaded {len(msdb_rxns)} MSDB reactions from disk in {time.time()-t:.1f}s '
          f'-- cached to {session.kbcache_dir}')

# Load the 100-model panel ids (the descriptive growth-model panel).
PANEL_IDS = open(RESULTS / 'selected_ids.txt').read().split()
print(f'Panel: {len(PANEL_IDS)} models')

# Load the original (no-rebound) growth from results.csv -- our 'on-disk FBA'
# reference.  This is the baseline that analyze_growth.py wrote.
ONDISK_FBA = {}
with open(RESULTS / 'results.csv') as fh:
    for row in csv.DictReader(fh):
        ONDISK_FBA[row['model_id']] = {
            'status'     : row['status'],
            'growth_flux': float(row['growth_flux']),
            'grows'      : row['grows'].lower() == 'true',
        }
print(f'On-disk FBA results loaded for {len(ONDISK_FBA)} models')

Loaded 56012 MSDB reactions from cache
Panel: 100 models
On-disk FBA results loaded for 5683 models


## 2. Baseline cascade (reproduces the upstream report)

Run `Estimate_Reaction_Reversibility.estimate_one` (via the
port in `reversibility_lib`) with `ReversibilityConfig()` --
every knob at its upstream default.  The cascade output should
match `Estimated_Reaction_Reversibility_Report_EQ.txt` exactly
modulo a small set of reactions whose stored `deltag` was
updated in MSDB after the report was regenerated.

In [3]:
# --- Baseline cascade ---------------------------------------------------
# Run the heuristic with ReversibilityConfig() defaults.  Caches the result
# so re-runs are instant.
CACHE_KEY_BASELINE = 'reversibility_baseline_v1'

def _run_baseline_cascade():
    rxns = copy.deepcopy(msdb_rxns)
    return lib.run_cascade(rxns, db_level='EQ', cfg=lib.ReversibilityConfig(),
                           gc_first=True)

try:
    baseline_cascade = session.cache.load(CACHE_KEY_BASELINE)
    print(f'baseline_cascade: loaded {len(baseline_cascade)} entries from cache')
except KeyError:
    t = time.time()
    baseline_cascade = _run_baseline_cascade()
    session.cache.save(CACHE_KEY_BASELINE, baseline_cascade, type_hint='dict',
                       metadata={'config': 'ReversibilityConfig() default'})
    print(f'baseline_cascade: {len(baseline_cascade)} reactions in {time.time()-t:.1f}s, cached')

BASELINE_MAP = {r: rev for r, (_, rev) in baseline_cascade.items()}

# --- Sanity check vs. the on-disk report ----------------------------------
# Compare our cascade's output to the existing
# Estimated_Reaction_Reversibility_Report_EQ.txt.  The MSDB JSON has been
# updated since the report was regenerated, so we expect a handful of
# reactions where the report says 'Incomplete' but our cascade now has data.
report = {}
with open('/scratch/ctaylor/ModelSEEDDatabase/Scripts/Thermodynamics/Estimated_Reaction_Reversibility_Report_EQ.txt') as fh:
    for line in fh:
        parts = line.rstrip('\n').split('\t')
        report[parts[0]] = (parts[1], parts[3])  # (status, new_rev)

agree = 0; disagree = []
for rxn, (status, rev) in baseline_cascade.items():
    rs = report.get(rxn)
    if rs is None:
        continue
    if rs == (status, rev):
        agree += 1
    else:
        disagree.append((rxn, rs, (status, rev)))

print(f'Cascade reproduces Estimated_Reaction_Reversibility_Report_EQ.txt:')
print(f'  exact matches:  {agree}/{len(baseline_cascade)}')
print(f'  mismatches:     {len(disagree)}  (expected drift from updated MSDB data)')
if disagree:
    print('  sample drift:')
    for s in disagree[:3]:
        print(f'    {s}')

baseline_cascade: loaded 56012 entries from cache


Cascade reproduces Estimated_Reaction_Reversibility_Report_EQ.txt:
  exact matches:  55737/56012
  mismatches:     275  (expected drift from updated MSDB data)
  sample drift:
    ('rxn00050', ('Incomplete (GCC)', '<'), ('MdeltaG(Min): 114.95', '<'))
    ('rxn00089', ('Incomplete', '?'), ('Incomplete (GCC)', '='))
    ('rxn00214', ('Incomplete (GCC)', '='), ('mMdeltaG: 0.00', '='))


## 3. Two FBA baselines on the 100-model panel

- **On-disk FBA** -- bounds untouched.  Should reproduce
  `results/results.csv` byte-for-byte (modulo solver jitter
  well below 1e-6).
- **Heuristic-baseline FBA** -- panel models rebound to the
  cascade's baseline reversibility map.  Diverges from on-disk
  wherever the model JSON was built from a different
  reversibility (template-time curation, pre-EQ heuristics,
  ...).  This is the reference point all variants are compared
  against -- it isolates "what does the heuristic say" from
  "what does the on-disk model say".

In [4]:
# --- Two FBA baselines --------------------------------------------------
# 1) on-disk FBA: bounds untouched, matches results.csv byte-for-byte.
# 2) heuristic-baseline FBA: rebind every panel model to BASELINE_MAP
#    (= MSDB cascade defaults) -- the reference point for variant heuristics.

CACHE_KEY_ONDISK = 'fba_ondisk_v1'
CACHE_KEY_HEURBASE = 'fba_heuristic_baseline_v1'

def _run_panel(rev_map, baseline_map=None):
    return gh.run_panel(PANEL_IDS, reversibility_map=rev_map,
                        baseline_map=baseline_map, n_workers=4)

try:
    ondisk_fba = session.cache.load(CACHE_KEY_ONDISK)
    print(f'ondisk_fba: loaded {len(ondisk_fba)} from cache')
except KeyError:
    t = time.time()
    ondisk_fba = _run_panel(rev_map=None)
    session.cache.save(CACHE_KEY_ONDISK, ondisk_fba, type_hint='dict',
                       metadata={'rebound': False})
    print(f'ondisk_fba: {len(ondisk_fba)} models in {time.time()-t:.1f}s')

# Sanity check: every panel model's status + flux matches results.csv.
mm = 0
for r in ondisk_fba:
    o = ONDISK_FBA[r['model_id']]
    if r['status'] != o['status'] or abs(r['growth_flux'] - o['growth_flux']) > 1e-6:
        mm += 1
print(f'on-disk FBA reproduces results.csv for panel: '
      f'{len(ondisk_fba) - mm}/{len(ondisk_fba)}  (mismatches: {mm})')

try:
    heur_baseline_fba = session.cache.load(CACHE_KEY_HEURBASE)
    print(f'heur_baseline_fba: loaded {len(heur_baseline_fba)} from cache')
except KeyError:
    t = time.time()
    heur_baseline_fba = _run_panel(rev_map=BASELINE_MAP)
    session.cache.save(CACHE_KEY_HEURBASE, heur_baseline_fba, type_hint='dict',
                       metadata={'rebound': True, 'map': 'BASELINE_MAP'})
    print(f'heur_baseline_fba: {len(heur_baseline_fba)} models in {time.time()-t:.1f}s')

# How much does the heuristic-baseline diverge from on-disk?
ondisk_idx = {r['model_id']: r for r in ondisk_fba}
n_grow_diff = sum(1 for r in heur_baseline_fba
                  if ondisk_idx[r['model_id']]['grows'] != r['grows'])
n_flux_diff = sum(1 for r in heur_baseline_fba
                  if abs(ondisk_idx[r['model_id']]['growth_flux']
                         - r['growth_flux']) > 1e-6)
print(f'heuristic-baseline vs on-disk: '
      f'{n_grow_diff} models flip grow-status, '
      f'{n_flux_diff} models have flux delta > 1e-6')

ondisk_fba: loaded 100 from cache
on-disk FBA reproduces results.csv for panel: 100/100  (mismatches: 0)
heur_baseline_fba: loaded 100 from cache
heuristic-baseline vs on-disk: 0 models flip grow-status, 98 models have flux delta > 1e-6


## 4. Heuristic variants

Each variant cell:

1. Builds a `ReversibilityConfig` with one knob flipped.
2. Recomputes the cascade and the per-reaction direction diff
   vs the baseline cascade.
3. Reruns FBA on the 100-model panel, but *only updates bounds
   for reactions whose direction changed* (so we measure the
   heuristic, not arbitrary differences between MSDB and the
   model JSONs).
4. Reports the panel growth diff.

Variants tagged `H*` are new suggestions added while writing
this notebook -- they are documented at the bottom of
`Reaction_Reversibility_Heuristics_Review.md`.

In [5]:
VARIANT_REGISTRY = {}

### Variant 3.1 -- Persist + use ln(reversibility_index)

*Heuristics Review § 2.1 / 3.1*

In [6]:
# -----------------------------------------------------------------------------
# Variant 3.1: Persist + use ln(reversibility_index)
# Heuristics Review § 2.1 / 3.1
# -----------------------------------------------------------------------------
def _cfg_3_1():
    ln_ri = lib.load_ln_reversibility_index()
    print(f'  loaded ln_RI for {len(ln_ri)} reactions from MetaNetX_Reaction_Energies.tbl')
    return lib.ReversibilityConfig(ln_ri_by_rxn=ln_ri)

def _build_variant_3_1():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_1()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_1_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_1()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.1'})
    print(f'variant 3.1: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.1: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_1_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.1'})
    print(f'variant 3.1: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.1: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.1'] = {
    'title'           : 'Persist + use ln(reversibility_index)',
    'section'         : '§ 2.1 / 3.1',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.1: 3256 reactions changed direction
  '<' -> '>':  66
  '=' -> '<':  1072
  '=' -> '>':  244
  '>' -> '<':  1874
variant 3.1: panel growth -- 75/100 models flip grow-status, 100 have flux delta > 1e-6


### Variant 3.3 -- Bennett-2009 per-metabolite concentration ranges

*Heuristics Review § 3.3*

In [7]:
# -----------------------------------------------------------------------------
# Variant 3.3: Bennett-2009 per-metabolite concentration ranges
# Heuristics Review § 3.3
# -----------------------------------------------------------------------------
def _cfg_3_3():
    return lib.ReversibilityConfig(
        per_met_conc_range=lib.BENNETT_2009_ECOLI,
        per_met_conc=lib.BENNETT_2009_MEAN,
    )

def _build_variant_3_3():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_3()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_3_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_3()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.3'})
    print(f'variant 3.3: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.3: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_3_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.3'})
    print(f'variant 3.3: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.3: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.3'] = {
    'title'           : 'Bennett-2009 per-metabolite concentration ranges',
    'section'         : '§ 3.3',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.3: 1316 reactions changed direction
  '<' -> '=':  30
  '=' -> '<':  214
  '=' -> '>':  974
  '>' -> '=':  98
variant 3.3: panel growth -- 0/100 models flip grow-status, 83 have flux delta > 1e-6


### Variant 3.3_wide -- Wider uniform concentration window [1e-7, 0.1] M

*Heuristics Review § 3.3 (fallback)*

In [8]:
# -----------------------------------------------------------------------------
# Variant 3.3_wide: Wider uniform concentration window [1e-7, 0.1] M
# Heuristics Review § 3.3 (fallback)
# -----------------------------------------------------------------------------
def _cfg_3_3_wide():
    return lib.ReversibilityConfig(cell_min=1e-7, cell_max=1e-1)

def _build_variant_3_3_wide():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_3_wide()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_3_wide_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_3_wide()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.3_wide'})
    print(f'variant 3.3_wide: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.3_wide: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_3_wide_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.3_wide'})
    print(f'variant 3.3_wide: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.3_wide: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.3_wide'] = {
    'title'           : 'Wider uniform concentration window [1e-7, 0.1] M',
    'section'         : '§ 3.3 (fallback)',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.3_wide: 1796 reactions changed direction
  '<' -> '=':  424
  '>' -> '=':  1372
variant 3.3_wide: panel growth -- 0/100 models flip grow-status, 12 have flux delta > 1e-6


### Variant 3.5 -- Per-reaction sigma band: k=1.96 (95%) instead of fixed +/-2 kcal

*Heuristics Review § 3.5*

In [9]:
# -----------------------------------------------------------------------------
# Variant 3.5: Per-reaction sigma band: k=1.96 (95%) instead of fixed +/-2 kcal
# Heuristics Review § 3.5
# -----------------------------------------------------------------------------
def _cfg_3_5():
    return lib.ReversibilityConfig(sigma_band_k=1.96)

def _build_variant_3_5():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_5()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_5_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_5()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.5'})
    print(f'variant 3.5: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.5: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_5_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.5'})
    print(f'variant 3.5: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.5: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.5'] = {
    'title'           : 'Per-reaction sigma band: k=1.96 (95%) instead of fixed +/-2 kcal',
    'section'         : '§ 3.5',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.5: 304 reactions changed direction
  '<' -> '=':  6
  '>' -> '=':  298
variant 3.5: panel growth -- 0/100 models flip grow-status, 0 have flux delta > 1e-6


### Variant 3.5_wide -- Per-reaction CC bound widening: k=1.96 on stored_bounds

*Heuristics Review § 3.5 / § 2.5*

In [10]:
# -----------------------------------------------------------------------------
# Variant 3.5_wide: Per-reaction CC bound widening: k=1.96 on stored_bounds
# Heuristics Review § 3.5 / § 2.5
# -----------------------------------------------------------------------------
def _cfg_3_5_wide():
    return lib.ReversibilityConfig(sigma_bounds_k=1.96)

def _build_variant_3_5_wide():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_5_wide()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_5_wide_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_5_wide()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.5_wide'})
    print(f'variant 3.5_wide: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.5_wide: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_5_wide_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.5_wide'})
    print(f'variant 3.5_wide: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.5_wide: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.5_wide'] = {
    'title'           : 'Per-reaction CC bound widening: k=1.96 on stored_bounds',
    'section'         : '§ 3.5 / § 2.5',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.5_wide: 546 reactions changed direction
  '<' -> '=':  106
  '>' -> '=':  440
variant 3.5_wide: panel growth -- 0/100 models flip grow-status, 0 have flux delta > 1e-6


### Variant 3.6 -- Drop the low-energy-compounds list entirely

*Heuristics Review § 3.6*

In [11]:
# -----------------------------------------------------------------------------
# Variant 3.6: Drop the low-energy-compounds list entirely
# Heuristics Review § 3.6
# -----------------------------------------------------------------------------
def _cfg_3_6():
    return lib.ReversibilityConfig(low_energy_cpds=())

def _build_variant_3_6():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_6()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_6_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_6()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.6'})
    print(f'variant 3.6: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.6: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_6_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.6'})
    print(f'variant 3.6: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.6: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.6'] = {
    'title'           : 'Drop the low-energy-compounds list entirely',
    'section'         : '§ 3.6',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.6: 3070 reactions changed direction
  '<' -> '=':  188
  '>' -> '=':  2882
variant 3.6: panel growth -- 0/100 models flip grow-status, 96 have flux delta > 1e-6


### Variant 3.7 -- Drop the CO2 1e-4 hardcoded concentration override

*Heuristics Review § 3.7*

In [12]:
# -----------------------------------------------------------------------------
# Variant 3.7: Drop the CO2 1e-4 hardcoded concentration override
# Heuristics Review § 3.7
# -----------------------------------------------------------------------------
def _cfg_3_7():
    return lib.ReversibilityConfig(apply_special_conc=False)

def _build_variant_3_7():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_7()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_7_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_7()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.7'})
    print(f'variant 3.7: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.7: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_7_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.7'})
    print(f'variant 3.7: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.7: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.7'] = {
    'title'           : 'Drop the CO2 1e-4 hardcoded concentration override',
    'section'         : '§ 3.7',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.7: 0 reactions changed direction
variant 3.7: panel growth -- 0/100 models flip grow-status, 0 have flux delta > 1e-6


### Variant 3.10_tight -- Tighten mMdeltaG band: +/-1 kcal/mol

*Heuristics Review § 3.10*

In [13]:
# -----------------------------------------------------------------------------
# Variant 3.10_tight: Tighten mMdeltaG band: +/-1 kcal/mol
# Heuristics Review § 3.10
# -----------------------------------------------------------------------------
def _cfg_3_10_tight():
    return lib.ReversibilityConfig(mm_band=1.0)

def _build_variant_3_10_tight():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_10_tight()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_10_tight_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_10_tight()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.10_tight'})
    print(f'variant 3.10_tight: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.10_tight: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_10_tight_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.10_tight'})
    print(f'variant 3.10_tight: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.10_tight: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.10_tight'] = {
    'title'           : 'Tighten mMdeltaG band: +/-1 kcal/mol',
    'section'         : '§ 3.10',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.10_tight: 7 reactions changed direction
  '=' -> '>':  7
variant 3.10_tight: panel growth -- 0/100 models flip grow-status, 0 have flux delta > 1e-6


### Variant 3.10_loose -- Loosen mMdeltaG band: +/-4 kcal/mol

*Heuristics Review § 3.10*

In [14]:
# -----------------------------------------------------------------------------
# Variant 3.10_loose: Loosen mMdeltaG band: +/-4 kcal/mol
# Heuristics Review § 3.10
# -----------------------------------------------------------------------------
def _cfg_3_10_loose():
    return lib.ReversibilityConfig(mm_band=4.0)

def _build_variant_3_10_loose():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_3_10_loose()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_3_10_loose_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_3_10_loose()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': '3.10_loose'})
    print(f'variant 3.10_loose: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant 3.10_loose: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_3_10_loose_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': '3.10_loose'})
    print(f'variant 3.10_loose: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant 3.10_loose: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['3.10_loose'] = {
    'title'           : 'Loosen mMdeltaG band: +/-4 kcal/mol',
    'section'         : '§ 3.10',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant 3.10_loose: 696 reactions changed direction
  '<' -> '=':  82
  '>' -> '=':  614
variant 3.10_loose: panel growth -- 0/100 models flip grow-status, 30 have flux delta > 1e-6


### Variant H1 -- (NEW) default direction = '?' instead of '=' for unresolved

*Heuristics Review § H1 (added by notebook)*

In [15]:
# -----------------------------------------------------------------------------
# Variant H1: (NEW) default direction = '?' instead of '=' for unresolved
# Heuristics Review § H1 (added by notebook)
# -----------------------------------------------------------------------------
def _cfg_H1():
    return lib.ReversibilityConfig(default_direction='?')

def _build_variant_H1():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_H1()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_H1_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_H1()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': 'H1'})
    print(f'variant H1: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant H1: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_H1_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': 'H1'})
    print(f'variant H1: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant H1: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['H1'] = {
    'title'           : "(NEW) default direction = '?' instead of '=' for unresolved",
    'section'         : '§ H1 (added by notebook)',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant H1: 6522 reactions changed direction
  '=' -> '?':  6522
variant H1: panel growth -- 0/100 models flip grow-status, 0 have flux delta > 1e-6


### Variant H2 -- (NEW) repair LOW_LOCAL_CONC shadow bug (O2/H2 at 1e-6 M)

*Heuristics Review § H2 (added by notebook)*

In [16]:
# -----------------------------------------------------------------------------
# Variant H2: (NEW) repair LOW_LOCAL_CONC shadow bug (O2/H2 at 1e-6 M)
# Heuristics Review § H2 (added by notebook)
# -----------------------------------------------------------------------------
def _cfg_H2():
    return lib.ReversibilityConfig(fix_low_local_conc=True)

def _build_variant_H2():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_H2()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_H2_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_H2()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': 'H2'})
    print(f'variant H2: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant H2: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_H2_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': 'H2'})
    print(f'variant H2: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant H2: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['H2'] = {
    'title'           : '(NEW) repair LOW_LOCAL_CONC shadow bug (O2/H2 at 1e-6 M)',
    'section'         : '§ H2 (added by notebook)',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant H2: 1 reactions changed direction
  '<' -> '=':  1
variant H2: panel growth -- 0/100 models flip grow-status, 0 have flux delta > 1e-6


### Variant H3 -- (NEW) repair phosphates shadow bug (ABC + low-E phosphate spread)

*Heuristics Review § H3 (added by notebook)*

In [17]:
# -----------------------------------------------------------------------------
# Variant H3: (NEW) repair phosphates shadow bug (ABC + low-E phosphate spread)
# Heuristics Review § H3 (added by notebook)
# -----------------------------------------------------------------------------
def _cfg_H3():
    return lib.ReversibilityConfig(fix_phosphates_shadow=True)

def _build_variant_H3():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_H3()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_H3_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_H3()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': 'H3'})
    print(f'variant H3: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant H3: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_H3_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': 'H3'})
    print(f'variant H3: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant H3: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['H3'] = {
    'title'           : '(NEW) repair phosphates shadow bug (ABC + low-E phosphate spread)',
    'section'         : '§ H3 (added by notebook)',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant H3: 1989 reactions changed direction
  '<' -> '=':  130
  '=' -> '<':  252
  '=' -> '>':  1209
  '>' -> '=':  398
variant H3: panel growth -- 21/100 models flip grow-status, 100 have flux delta > 1e-6


### Variant H4 -- (NEW) stack 3.1 + 3.5 + Bennett: 'best-evidence' composite

*Heuristics Review § H4 (added by notebook)*

In [18]:
# -----------------------------------------------------------------------------
# Variant H4: (NEW) stack 3.1 + 3.5 + Bennett: 'best-evidence' composite
# Heuristics Review § H4 (added by notebook)
# -----------------------------------------------------------------------------
def _cfg_H4():
    ln_ri = lib.load_ln_reversibility_index()
    return lib.ReversibilityConfig(
        ln_ri_by_rxn=ln_ri,
        sigma_band_k=1.96,
        per_met_conc_range=lib.BENNETT_2009_ECOLI,
        per_met_conc=lib.BENNETT_2009_MEAN,
    )

def _build_variant_H4():
    rxns = copy.deepcopy(msdb_rxns)
    cfg = _cfg_H4()
    return lib.run_cascade(rxns, db_level='EQ', cfg=cfg, gc_first=True)

CACHE_KEY = 'reversibility_variant_H4_v1'
try:
    variant = session.cache.load(CACHE_KEY)
except KeyError:
    t = time.time()
    variant = _build_variant_H4()
    session.cache.save(CACHE_KEY, variant, type_hint='dict',
                       metadata={'variant': 'H4'})
    print(f'variant H4: cascade in {time.time()-t:.1f}s')

VARIANT_MAP = {r: rev for r, (_, rev) in variant.items()}
diff = gh.reversibility_diff(BASELINE_MAP, VARIANT_MAP)
print(f'variant H4: {diff["n_changed"]} reactions changed direction')
for transition, n in sorted(diff['by_transition'].items()):
    print(f'  {transition[0]!r} -> {transition[1]!r}:  {n}')

FBA_KEY = 'fba_variant_H4_v2'
try:
    variant_fba = session.cache.load(FBA_KEY)
except KeyError:
    t = time.time()
    # Fully rebind to VARIANT_MAP (no baseline filter) -- this makes the
    # variant_fba bounds directly comparable to heur_baseline_fba bounds.
    # Reactions where VARIANT_MAP == BASELINE_MAP get identical bounds in
    # both runs; only reactions where the variant differs contribute to
    # the FBA diff.
    variant_fba = gh.run_panel(PANEL_IDS, reversibility_map=VARIANT_MAP,
                                baseline_map=None, n_workers=4)
    session.cache.save(FBA_KEY, variant_fba, type_hint='dict',
                       metadata={'variant': 'H4'})
    print(f'variant H4: panel FBA in {time.time()-t:.1f}s')

fba_diff = gh.diff_panel(heur_baseline_fba, variant_fba)
print(f'variant H4: panel growth -- '
      f'{fba_diff["n_grow_change"]}/{fba_diff["n_models"]} models flip grow-status, '
      f'{fba_diff["n_flux_change"]} have flux delta > 1e-6')

# Register the variant for the cross-variant summary table.
VARIANT_REGISTRY['H4'] = {
    'title'           : "(NEW) stack 3.1 + 3.5 + Bennett: 'best-evidence' composite",
    'section'         : '§ H4 (added by notebook)',
    'rev_diff'        : diff,
    'fba_diff'        : fba_diff,
}

variant H4: 3317 reactions changed direction
  '<' -> '=':  32
  '<' -> '>':  54
  '=' -> '<':  990
  '=' -> '>':  1118
  '>' -> '<':  850
  '>' -> '=':  273
variant H4: panel growth -- 65/100 models flip grow-status, 99 have flux delta > 1e-6


## 5. Cross-variant summary

In [19]:
# Cross-variant summary table.
import pandas as pd
rows = []
rows.append({
    'variant': 'baseline',
    'section': '(reference)',
    'title'  : 'ReversibilityConfig() default',
    'rxns_changed_vs_baseline'  : 0,
    'panel_models_flux_changed' : 0,
    'panel_models_grow_flipped' : 0,
})
for tag, info in VARIANT_REGISTRY.items():
    rows.append({
        'variant': tag,
        'section': info['section'],
        'title'  : info['title'],
        'rxns_changed_vs_baseline'  : info['rev_diff']['n_changed'],
        'panel_models_flux_changed' : info['fba_diff']['n_flux_change'],
        'panel_models_grow_flipped' : info['fba_diff']['n_grow_change'],
    })
summary = pd.DataFrame(rows).set_index('variant')
print('Cross-variant summary:')
display(summary)

# Persist for later inspection / paper figures.
session.cache.save('reversibility_variant_summary_v1', summary,
                   type_hint='dataframe',
                   metadata={'panel': 'selected_ids.txt (100 models)'})

Cross-variant summary:


,section,title,rxns_changed_vs_baseline,panel_models_flux_changed,panel_models_grow_flipped
variant,,,,,
baseline,(reference),ReversibilityConfig() default,0,0,0
3.1,§ 2.1 / 3.1,Persist + use ln(reversibility_index),3256,100,75
3.3,§ 3.3,Bennett-2009 per-metabolite concentration ranges,1316,83,0
3.3_wide,§ 3.3 (fallback),"Wider uniform concentration window [1e-7, 0.1] M",1796,12,0
3.5,§ 3.5,Per-reaction sigma band: k=1.96 (95%) instead ...,304,0,0
3.5_wide,§ 3.5 / § 2.5,Per-reaction CC bound widening: k=1.96 on stor...,546,0,0
3.6,§ 3.6,Drop the low-energy-compounds list entirely,3070,96,0
3.7,§ 3.7,Drop the CO2 1e-4 hardcoded concentration over...,0,0,0
3.10_tight,§ 3.10,Tighten mMdeltaG band: +/-1 kcal/mol,7,0,0


CacheEntry(id='reversibility_variant_summary_v1', type='dataframe', blob_path='blobs/d6437dfba07f0aca7c5ef1264b92fd214cad80331f173b2fc55de6047a3b5099.parquet', content_hash='d6437dfba07f0aca7c5ef1264b92fd214cad80331f173b2fc55de6047a3b5099', n_bytes=5734, metadata={'panel': 'selected_ids.txt (100 models)', 'shape': [14, 5], 'columns': ['section', 'title', 'rxns_changed_vs_baseline', 'panel_models_flux_changed', 'panel_models_grow_flipped']}, created_at=datetime.datetime(2026, 6, 3, 21, 15, 39, 288701, tzinfo=datetime.timezone.utc))